### <u><center><span style="color:darkorchid">ChatGPT</span> Resume Optimizer Final</u></center>

#### This will be the deployable version of the <span style="color:orange">original</span> <span style="color:darkorchid">ChatGPT </span><span style="color:orange">notebook</span>. The main differences here are:
> ##### 1.) Instead of creating paths to my personal machine, the program will create paths to the user's own machine; and
> ##### 2.) The back-end of the button <span style="color:deeppink">Company Name</span> will be created here as well. The point of this button is to include the hiring company's name in the title of your saved resume for facilitated resume tracking.

#### <u><span style="color:green">Imports</span></u>:
##### Because we're making the changes listed above, two additional Python librarys are imported in this section: <b>uuid</b> and <b>re</b>:
> ##### <b><u>uuid</u></b> - generates unique IDs so multiple users can upload files and download tailored resumes with unique filenames.
> ##### <b><u>re</u></b> - the regular expressions library lets us take the company name from the one the user entered and removes all unnecessary spaces for a cleaner filename.

In [1]:
import os # use personal openai key
import uuid # for unique file usernames
import re # for future 'company name' button
from dotenv import load_dotenv # loads environmental variables

# for working with OpenAI:
from openai import OpenAI

# HTML to Markdown for editing
from markdown import markdown
from IPython.display import display, Markdown # <-- make it look nice in the notebook

# for design
from weasyprint import HTML # see previous notebook for weasyprint notes

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
# load the environment variables
load_dotenv()

True

In [3]:
# make a destination 'gpt_resumes' directory for the work
os.makedirs("gpt_resumes", exist_ok=True)

***
#### Step 1: Open and Read the Markdown Resume
***

In [6]:
file_path = input("Please paste the full path to your resume file (eg: '/Users/your_name/Desktop/your_resume.md'): ").strip()

with open(file_path, "r", encoding = "utf-8") as resume_file:
    resume_string = resume_file.read()

Please paste the full path to your resume file (eg: '/Users/your_name/Desktop/your_resume.md'):  /Users/nicholasjoseph/Desktop/nj_master_resume.md


***
#### Step 2: Input the Desired Job Description
***

In [7]:
job_desc_string = input("Please paste selected job descripton here: ")

Please paste selected job descripton here:  About the job We are the world's leading Next-Gen Mobile App security company.  Promon operates at the center of innovation, emerging trends, and high-end mobile app security solutions. Some of the world's largest enterprises trust us to secure their apps from cyber threats. Working together, our global team makes the digital world safer and more secure, one app at a time.  With billions of mobile end points to protect, now is a fantastic time to join Promon. You'll be part of a vibrant culture and help drive our ambitious plans for the years ahead.  Our headquarters is based in Oslo, Norway, and with our global footprint, we have employees and customers all over the world.  Your Role And Impact  Ready to play a key role in driving technical sales success across EMEA? We’re looking for a skilled Sales Engineer to support our fast-growing direct and partner-led sales motion. You’ll work closely with prospects, customers, and partners to delive

***
#### Step 3: Input the Company Name and "Slug" It
> ##### Strip whitespace, get rid of stuff like apostrophes, and lowercase everything
> ##### Turns "Bill and Ted's Excellent Adventure Company" into "billandtedsexcellentadventurecompany"
> ##### Again, so user can better track their resumes
***

In [8]:
# strip whitespace 
company_raw = input("Please enter the company's name: ").strip()
# cut out anything that's not a letter or number from the company name and lowercase everything
company_slugged = re.sub(r"[^a-zA-Z0-9_-]", "", company_raw.lower())

Please enter the company's name:  Promon


***
#### Step 4: Template Some Prompt Engineering
> ##### Because ChatGPT can talk to an ATS better than we can
***

In [9]:
prompt_template = lambda resume_string, job_desc_string : f"""
### Role: 
You are a professional resume optimization expert, tailoring my resume to fit specific job descriptions. 
You know my job preferences include collaborating with people, and helping businesses get the most out of their data.
Your goal is to optimize my resume and provide actionable suggestions for improvement to align with the target role.

### Guidelines:
1. **Relevance**:
    - Prioritize the particular skills and experiences I have with what is **most relevant to the job position**.
    - De-emphasize or even completely remove irrelevant details to ensure a **concise** and **targeted** resume.
    - Limit work experience section to 2-3 most relevant roles
    - Limit bullet points under each role to 2-3 most relevant impacts
    - Select only the core competencies most relevant to the job description

2. **Action-Driven Results**:
    - Choose **strong action verbs** and **quantifiable results** (eg: percentages, revenues, efficiency improvement, etc.)

3. **Summary Selection**:
    - Please tailor the best best Summary format for the job description and Recruiter expectations.

4. **Keyword Optimization**:
    - Integrate **keywords** and phrases from the job description naturally to optimize for Applicant Tracking Systems (ATS)

5. **Additional Suggestions*** *(if gaps exist)*:
    - If the resume does not fully align with the job description, suggest:
        a.) **Additional technical or soft skills** that I could add to make my profile stronger.
        b.) **Certifications or courses** I have (or could pursue) that would bridge the gap(s).
        c.) **Project ideas or experiences** that would better align with the role.
        d.) **Compare intent** of job description with selected Summary and provide ideas to better tailor the Summary.
        e.) **Score** predicted match between resume and job description.

6. **Formatting**:
    - Ouptut the tailored resume in **clean Markdown format**.
    - Include an **"Additional Suggestions"** section at the end with actionable improvement recommendations.

---

## Input:
- **My resume**:
{resume_string}

- **The Job Description**:
{job_desc_string}

---

### Output:
1. **Tailored Resume**:
    - A resume in **Markdown format** that emphasizes relevant experience, skills, and achievements.
    - Incorporates job description **keywords** to optimize for ATS.
    - Uses confident language and is no longer than **one page**.

2. **Additional Suggestions** *(if applicable)*:
    - List **skills** that could strengthen alignment with the role.
    - Recommend **certifications or courses** to pursue.
    - Suggest **specific projects or experiences** to develop.
"""

# assign it a variable for use later on in the code
prompt = prompt_template(resume_string, job_desc_string)

***
#### Step 5: Generate the Resume with <b><span style="color:darkorchid">gpt-4o-mini</span></b>:
> ##### Sets up the api client, desginates the model, outlines the roles, and balances response creativity with accuracy.
> ##### More detailed notes available in original <b><span style="color:firebrick">rez_optimizer_chatgpt</span></b> notebook
***

In [10]:
# set up client
open_apikey = os.environ.get("openapi_apikey")
    
client = OpenAI(api_key = open_apikey)

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    
    messages = [
        {"role" : "system", "content" : "Expert Resume Writer"},
        {"role" : "user", "content" : prompt}
    ],
     
    temperature = 0.7
)

# Get our response
response_string = response.choices[0].message.content

***
#### Step 6: Display the <span style="color:darkorchid">ChatGPT</span>-Optimized Resume
***

In [11]:
response_list = response_string.split("## Additional Suggestions")

display(Markdown(response_list[0]))

## Nicholas Joseph
**Product Manager | Servant Leader | Tech Enthusiast**

Email: [nickpjoseph210@gmail.com](mailto:nickpjoseph210@gmail.com) | Phone: (210) 771-9853 | LinkedIn: [nickjoseph](https://www.linkedin.com/in/nickjoseph) | [GitHub](https://github.com/npj210mlk)

### Career Summary  
Results-driven Product Manager with extensive experience in data engineering and project management. Proven track record of delivering innovative solutions in fast-paced environments, particularly within the B2B software and cybersecurity sectors. Adept at engaging customers, leading technical discovery, and translating complex technical concepts into business value. Strong collaborator with a focus on driving efficiencies and ensuring project success through robust stakeholder engagement.

### Relevant Experience  

**<u>Cloud Data Engineer**  
Data Clymer (now Spaulding Ridge), Remote | 07/2022 - 06/2023</u>  
- Led technical discovery and solution mapping for ‘Fan 360’ and ‘Golden Record’ initiatives, enhancing fan engagement and operational efficiencies for NFL and MLB by reducing data collection intervals from 24 hours to 15 minutes.
- Collaborated with cross-functional teams to architect and implement data pipelines that improved reporting accuracy by 53%, facilitating informed decision-making.
- Delivered technical presentations and knowledge transfer sessions to partners, enhancing sales teams' capabilities and ensuring alignment with customer needs.

**<u>Data Engineer**  
Gathi Analytics (now Apexon), Remote | 12/2020 - 03/2022</u>  
- Transformed legacy banking data systems to cloud-based architecture, improving data accessibility and compliance with regulatory standards.
- Developed interactive dashboards and reports for stakeholders, presenting critical findings that supported strategic business decisions.
- Engaged with clients to ensure data integrity and compliance, effectively communicating technical solutions in business terms.

### Skills  
**Technical Skills:**  
- Programming: Python, SQL, JavaScript (familiarity), API Integration, Cloud Data Warehousing (Snowflake, Azure, GCP)  
- Data Engineering: Data Pipeline Architecture, Data Transformation Tools (dbt, Matillion), Data Visualization (Tableau)  
- Security Standards: Familiarity with OWASP MASVS, NIST, and ISO 27001  

**Soft Skills:**  
- Stakeholder Engagement, Cross-functional Collaboration, Clear Communication, Problem-solving, Mentoring, Technical Storytelling  

### Education  
**Certificate of Completion** - PMP Exam Prep - 35 PDU Technical Institute of America, Remote | 2024  
**Certificate of Completion** - Data Science Codeup, San Antonio, TX | 2020  
**Bachelor's** - Biology Concordia University, Austin, TX | 2001  

---

#

***
#### Step 7: Save the Resume In PDF and Markdown Using <span style="color:deeppink">Weasyprint</span> and <b>uuid</b>
***

In [12]:
# 8-character unique id from uuid
unique_id = str(uuid.uuid4())[:8]
# Save and send files to gpt_resumes directory
output_pdf_file = f"gpt_resumes/{company_slugged}{unique_id}_optimized.pdf"
output_markdown_file = f"gpt_resumes/{company_slugged}{unique_id}_optimized.md"

In [13]:
# Convert / save response to HTML so the web can read it using Weasyprint
html_content = markdown(response_list[0])
HTML(string=html_content).write_pdf(output_pdf_file)

In [14]:
# Write and save as Markdown for ease of editing
with open(output_markdown_file, "w", encoding = "utf-8") as file:
    file.write(response_list[0])

***
#### Step 7: Display Suggestions for Improvement Here for Easy Reference While Editing
> ##### <span style="color:red">Recall</span>: we split the '<span style="font-family:menlo"><b>response_list</span></b>' variable into two parts, [0] - the resume, and [1] - suggestions for improvement
***

In [16]:
display(Markdown(response_list[1]))


1. **Skills to Strengthen Alignment**:
   - Mobile Development Knowledge: Consider expanding proficiency in mobile application development frameworks (e.g., Kotlin, Obj-C).
   - Cybersecurity Fundamentals: Familiarize yourself with mobile threat landscapes and security frameworks to enhance relevance to the role.

2. **Certifications or Courses**:
   - Cybersecurity Certifications: Courses like CompTIA Security+, Certified Information Systems Security Professional (CISSP), or Certified Ethical Hacker (CEH) could be beneficial.
   - Mobile App Security Training: Look for specialized training programs focused on mobile app security best practices.

3. **Project Ideas**:
   - Develop a personal project or contribute to open-source projects focused on mobile security to showcase your skills in a practical context.
   - Create a case study on implementing security measures in a mobile app, detailing the process and outcomes, which can be used in discussions with potential employers.